# Module 091: Concurrency: Threading

Phase 10: Concurrency & Internals | Duration: 2.5 hours

## Thread Basics: Start and Join

In [ ]:
import threading
import time
from typing import List

def worker(name: str, delay: float) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {name} started")
    time.sleep(delay)
    print(f"[{time.strftime('%H:%M:%S')}] {name} finished")

threads: List[threading.Thread] = []
for i in range(3):
    t = threading.Thread(target=worker, args=(f"Thread-{i}", 0.5 * (i + 1)))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("All threads completed")

## Daemon Threads

In [ ]:
import threading
import time

def background_poll() -> None:
    count: int = 0
    while True:
        count += 1
        print(f"Daemon polling... ({count})")
        time.sleep(0.5)

daemon: threading.Thread = threading.Thread(target=background_poll, daemon=True)
daemon.start()

time.sleep(2)
print("Main thread exiting; daemon will be killed")

## Thread Safety and Locks

In [ ]:
import threading
from typing import List

counter: int = 0
lock: threading.Lock = threading.Lock()
ITERATIONS: int = 10000

def unsafe_increment() -> None:
    global counter
    for _ in range(ITERATIONS):
        counter += 1

def safe_increment() -> None:
    global counter
    for _ in range(ITERATIONS):
        with lock:
            counter += 1

counter = 0
threads = [threading.Thread(target=unsafe_increment) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"Unsafe result (race condition): {counter} (expected {ITERATIONS * 5})")

counter = 0
threads = [threading.Thread(target=safe_increment) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"Safe result (locked): {counter} (expected {ITERATIONS * 5})")

## Queue: Thread-Safe Communication

In [ ]:
import threading
import time
from queue import Queue
from typing import List

def producer(q: Queue, items: List[str]) -> None:
    for item in items:
        q.put(item)
        print(f"Produced: {item}")
        time.sleep(0.2)

def consumer(q: Queue, name: str) -> None:
    while True:
        item = q.get()
        if item is None:
            break
        print(f"{name} consumed: {item}")
        q.task_done()

q: Queue = Queue()
prod = threading.Thread(target=producer, args=(q, [f"Task-{i}" for i in range(5)]))
cons = threading.Thread(target=consumer, args=(q, "Worker-1"))

prod.start()
cons.start()
prod.join()
q.put(None)  # Sentinel
cons.join()
print("All items processed")

## GIL Demonstration

The GIL prevents true parallel execution of Python bytecode. CPU-bound threads actually run sequentially.

In [ ]:
import threading
import time
from typing import List

def cpu_heavy(n: int) -> None:
    total: int = 0
    for i in range(n):
        total += i ** 2
    print(f"Result: {total}")

N: int = 50000000

# Sequential
start: float = time.time()
cpu_heavy(N)
cpu_heavy(N)
sequential_time: float = time.time() - start
print(f"Sequential: {sequential_time:.2f}s")

# Threaded (no speedup due to GIL)
start = time.time()
threads = [threading.Thread(target=cpu_heavy, args=(N,)) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threaded_time: float = time.time() - start
print(f"Threaded: {threaded_time:.2f}s")

## GIL Diagram

```
Time ─────────────────────────────────────────►
     Thread 1  ████░░░░████░░░░████░░░░████
     Thread 2  ░░░░████░░░░████░░░░████░░░░
     GIL held  ████░░░░████░░░░████░░░░████
                ^── Only one thread runs at a time ──►
```

The GIL ensures only one thread executes Python bytecode at any moment.
I/O-bound threads benefit because they release the GIL during I/O waits.